Modifications from original:

Dataset: yelp_review_full instead of WikiText-2

Tokenizer: roberta-base (byte-level BPE) instead of GPT-2

Block size: 512 tokens instead of 128

Batch size: 32 instead of 8

Added: token length statistics cell (avg, min, max) before chunking

Added: torch.equal(ids, labels) verification check in final cell

Used add_special_tokens=False explicitly during tokenization

Decode sanity check on 3rd sequence instead of 1st

In [ ]:
!pip install transformers datasets torch

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
import torch

In [ ]:
# 1. Load the Yelp Review Full dataset (5-class sentiment, ~650k reviews)
# Only the review 'text' field is used — we ignore the star-rating label
dataset = load_dataset("yelp_review_full", split="train")
print(f"Training examples : {len(dataset)}")
print(f"Column names      : {dataset.column_names}")
print(f"\nSample review:\n{dataset[2]['text']}")

In [ ]:
# 2. Use RoBERTa's tokenizer — byte-level BPE, case-sensitive, no [CLS]/[SEP] by default
# This is different from both GPT-2 (causal) and DistilBERT (WordPiece, lowercased)
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
print(f"Vocab size        : {tokenizer.vocab_size}")
print(f"Model max length  : {tokenizer.model_max_length}")
print(f"Special tokens    : {tokenizer.all_special_tokens}")

In [ ]:
# 3. Tokenize the dataset in batches using .map()
# add_special_tokens=False so we don't insert <s>/</s> between every article
def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=False,
        add_special_tokens=False
    )

tokenized_ds = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text", "label"]
)

# Quick sanity check: decode a sample back to readable text
sample_ids = tokenized_ds[5]["input_ids"]
print(f"Columns           : {tokenized_ds.column_names}")
print(f"Example token count: {len(sample_ids)}")
print(f"First 20 token IDs: {sample_ids[:20]}")
print(f"Decoded           : {tokenizer.decode(sample_ids[:20])}")

In [ ]:
# 4. Compute token-length statistics across the dataset before chunking
# Helpful to decide an appropriate block size
lengths = [len(ex["input_ids"]) for ex in tokenized_ds]
avg_len = sum(lengths) / len(lengths)
print(f"Avg tokens per review : {avg_len:.1f}")
print(f"Min / Max             : {min(lengths)} / {max(lengths)}")
print(f"Reviews under 512 tok : {sum(l <= 512 for l in lengths) / len(lengths):.1%}")

In [ ]:
# 5. Concatenate all token sequences then slice into fixed-length blocks of 512 tokens
# Concatenating first avoids wasting capacity on short sequences
BLOCK_SIZE = 512

def chunk_into_blocks(examples):
    all_ids   = sum(examples["input_ids"], [])
    all_masks = sum(examples["attention_mask"], [])

    # Drop the remainder so every block is exactly BLOCK_SIZE
    keep = (len(all_ids) // BLOCK_SIZE) * BLOCK_SIZE
    all_ids, all_masks = all_ids[:keep], all_masks[:keep]

    chunked_ids   = [all_ids[i : i + BLOCK_SIZE]   for i in range(0, keep, BLOCK_SIZE)]
    chunked_masks = [all_masks[i : i + BLOCK_SIZE] for i in range(0, keep, BLOCK_SIZE)]

    return {"input_ids": chunked_ids, "attention_mask": chunked_masks}

lm_ds = tokenized_ds.map(chunk_into_blocks, batched=True, batch_size=500)
print(f"Total {BLOCK_SIZE}-token chunks : {len(lm_ds)}")

In [ ]:
# 6. Wrap in a PyTorch DataLoader — batch size 32, shuffled
def collate_fn(samples):
    ids   = torch.tensor([s["input_ids"]      for s in samples], dtype=torch.long)
    masks = torch.tensor([s["attention_mask"] for s in samples], dtype=torch.long)
    # For causal LM the model takes labels = input_ids and shifts internally
    return {"input_ids": ids, "attention_mask": masks, "labels": ids.clone()}

train_loader = DataLoader(
    lm_ds,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)
print(f"Total batches : {len(train_loader)}")

In [ ]:
# 7. Pull one batch and verify shapes, then decode a sequence back to text
for batch in train_loader:
    ids, masks, labels = batch["input_ids"], batch["attention_mask"], batch["labels"]
    print(f"input_ids shape    : {ids.shape}")
    print(f"attention_mask     : {masks.shape}")
    print(f"labels shape       : {labels.shape}")
    print(f"dtype              : {ids.dtype}")

    # Verify labels match input_ids (should be True)
    print(f"labels == input_ids: {torch.equal(ids, labels)}")

    # Decode the third sequence in the batch for a human-readable check
    print(f"\nDecoded sample (seq 2, first 60 tokens):\n{tokenizer.decode(ids[2][:60])}")
    break